# DBSCAN: Latihan Interaktif

Notebook ini berisi latihan hands-on untuk memahami DBSCAN clustering.

Lihat `README.md` untuk teori lengkap.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons

from implementation import DBSCAN, generate_clustering_data, plot_dbscan_results

np.random.seed(42)
print("✓ Setup complete!")

## Latihan 1: Basic DBSCAN

In [ ]:
# Generate store location data
X = generate_clustering_data(n_samples=300, random_state=42)

# Fit DBSCAN
dbscan = DBSCAN(eps=1.5, min_samples=5)
labels = dbscan.fit_predict(X)

print(f"Clusters found: {dbscan.n_clusters_}")
print(f"Noise points: {np.sum(labels == -1)}")
print(f"Core samples: {len(dbscan.core_sample_indices_)}")

# Plot
fig = plot_dbscan_results(X, labels, dbscan.core_sample_indices_)
plt.show()

## Latihan 2: Parameter Tuning - eps

In [ ]:
# Test different eps values
eps_values = [0.5, 1.0, 1.5, 2.0, 3.0]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, eps in enumerate(eps_values):
    dbscan = DBSCAN(eps=eps, min_samples=5)
    labels = dbscan.fit_predict(X)
    
    ax = axes[idx]
    unique_labels = set(labels)
    colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))
    
    for label, color in zip(unique_labels, colors):
        if label == -1:
            color = [0, 0, 0, 1]
            marker = 'x'
        else:
            marker = 'o'
        mask = labels == label
        ax.scatter(X[mask, 0], X[mask, 1], c=[color], marker=marker, s=30, alpha=0.7)
    
    n_clusters = dbscan.n_clusters_
    n_noise = np.sum(labels == -1)
    ax.set_title(f'eps={eps}\n{n_clusters} clusters, {n_noise} noise', fontweight='bold')
    ax.grid(True, alpha=0.3)

axes[-1].axis('off')
plt.tight_layout()
plt.show()

print("Observation: Smaller eps → more clusters & noise")

## Latihan 3: Parameter Tuning - min_samples

In [ ]:
# Test different min_samples values
min_samples_values = [2, 5, 10, 15]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, min_samples in enumerate(min_samples_values):
    dbscan = DBSCAN(eps=1.5, min_samples=min_samples)
    labels = dbscan.fit_predict(X)
    
    ax = axes[idx]
    unique_labels = set(labels)
    colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))
    
    for label, color in zip(unique_labels, colors):
        if label == -1:
            color = [0, 0, 0, 1]
            marker = 'x'
        else:
            marker = 'o'
        mask = labels == label
        ax.scatter(X[mask, 0], X[mask, 1], c=[color], marker=marker, s=40, alpha=0.7)
    
    n_clusters = dbscan.n_clusters_
    n_noise = np.sum(labels == -1)
    ax.set_title(f'min_samples={min_samples}\n{n_clusters} clusters, {n_noise} noise', fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Observation: Larger min_samples → fewer clusters, more noise")

## Latihan 4: DBSCAN on Non-Spherical Data

In [ ]:
# Create moon-shaped data
X_moons, _ = make_moons(n_samples=300, noise=0.05, random_state=42)

# DBSCAN
dbscan = DBSCAN(eps=0.2, min_samples=5)
labels = dbscan.fit_predict(X_moons)

# K-Means for comparison
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=2, random_state=42)
kmeans_labels = kmeans.fit_predict(X_moons)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# DBSCAN
ax = axes[0]
unique_labels = set(labels)
colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))
for label, color in zip(unique_labels, colors):
    if label == -1:
        color = [0, 0, 0, 1]
        marker = 'x'
    else:
        marker = 'o'
    mask = labels == label
    ax.scatter(X_moons[mask, 0], X_moons[mask, 1], c=[color], marker=marker, s=50, alpha=0.7)
ax.set_title('DBSCAN: Handles non-spherical shapes', fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3)

# K-Means
ax = axes[1]
scatter = ax.scatter(X_moons[:, 0], X_moons[:, 1], c=kmeans_labels, cmap='Spectral', s=50, alpha=0.7)
ax.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
          c='red', marker='X', s=200, edgecolors='black', linewidth=2)
ax.set_title('K-Means: Struggles with non-spherical', fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ DBSCAN correctly identifies both moons!")

## Latihan 5: K-Distance Graph (Finding Optimal eps)

In [ ]:
from sklearn.neighbors import NearestNeighbors

# Use original store data
X = generate_clustering_data(n_samples=300, random_state=42)

# K-distance graph
k = 5  # min_samples
nbrs = NearestNeighbors(n_neighbors=k).fit(X)
distances, indices = nbrs.kneighbors(X)

# Sort k-th nearest neighbor distances
k_dist = np.sort(distances[:, k-1], axis=0)[::-1]

# Plot
plt.figure(figsize=(10, 6))
plt.plot(k_dist, linewidth=2)
plt.axhline(y=1.5, color='r', linestyle='--', label='Suggested eps=1.5')
plt.ylabel(f'{k}-th Nearest Neighbor Distance', fontsize=12)
plt.xlabel('Points sorted by distance', fontsize=12)
plt.title('K-Distance Graph (Find the Elbow)', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Look for the 'elbow' where the curve sharply increases.")
print("That distance is a good candidate for eps parameter.")

## Latihan 6: Analyzing Results

In [ ]:
# Fit with optimal params
dbscan = DBSCAN(eps=1.5, min_samples=5)
labels = dbscan.fit_predict(X)

# Analyze each cluster
print("Cluster Analysis:")
print("=" * 50)

for cluster_id in range(dbscan.n_clusters_):
    mask = labels == cluster_id
    cluster_points = X[mask]
    
    # Compute cluster statistics
    center = cluster_points.mean(axis=0)
    std = cluster_points.std(axis=0)
    n_points = len(cluster_points)
    
    print(f"\nCluster {cluster_id}:")
    print(f"  Size: {n_points} stores")
    print(f"  Center: ({center[0]:.2f}, {center[1]:.2f})")
    print(f"  Std: ({std[0]:.2f}, {std[1]:.2f})")

# Noise analysis
noise_mask = labels == -1
n_noise = np.sum(noise_mask)
print(f"\nNoise Points: {n_noise}")
if n_noise > 0:
    noise_points = X[noise_mask]
    print(f"  These are isolated stores that don't fit into dense clusters.")

print("\n" + "=" * 50)